In [1]:
from langchain_classic.document_loaders import TextLoader
from langchain_core.prompts import PromptTemplate
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import WikipediaLoader

c:\Users\Sakshi Sinha\Downloads\RAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
#Load and chunk your document
loader = WikipediaLoader(
    query="Founder of Alphabet company",
    load_max_docs = 5
)

documents = loader.load()
documents

[Document(metadata={'title': 'Alphabet Inc.', 'summary': 'Alphabet Inc. is an American multinational technology conglomerate holding company headquartered in Mountain View, California. It was created through a restructuring of Google on October 2, 2015, and became the parent holding company of Google and several former Google subsidiaries. Alphabet is listed on the large-cap section of the Nasdaq under the ticker symbols GOOGL and GOOG; both classes of stock are components of major stock market indices such as the S&P 500 and Nasdaq-100. Alphabet has been described as a Big Tech company.\nThe establishment of Alphabet Inc. was prompted by a desire to make the core Google business "cleaner and more accountable" while allowing greater autonomy to group companies that operate in businesses other than Internet services. Founders Larry Page and Sergey Brin announced their resignation from their executive posts in December 2019, with the CEO role to be filled by Sundar Pichai, who is also th

In [3]:
#Chunking the documents 
splitter = RecursiveCharacterTextSplitter(chunk_size = 1000, chunk_overlap = 100)
chunks = splitter.split_documents(documents)
chunks

[Document(metadata={'title': 'Alphabet Inc.', 'summary': 'Alphabet Inc. is an American multinational technology conglomerate holding company headquartered in Mountain View, California. It was created through a restructuring of Google on October 2, 2015, and became the parent holding company of Google and several former Google subsidiaries. Alphabet is listed on the large-cap section of the Nasdaq under the ticker symbols GOOGL and GOOG; both classes of stock are components of major stock market indices such as the S&P 500 and Nasdaq-100. Alphabet has been described as a Big Tech company.\nThe establishment of Alphabet Inc. was prompted by a desire to make the core Google business "cleaner and more accountable" while allowing greater autonomy to group companies that operate in businesses other than Internet services. Founders Larry Page and Sergey Brin announced their resignation from their executive posts in December 2019, with the CEO role to be filled by Sundar Pichai, who is also th

In [4]:
#Build vectorstore
vector_store = FAISS.from_documents(
    documents= chunks,
    embedding= HuggingFaceEmbeddings(model_name = "all-MiniLM-L6-v2")
)
vector_store

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4427.55it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [15]:
retriever = vector_store.as_retriever(
    search_type="mmr", search_kwargs={"k": 4, "lambda_mult": 0.7}
)

In [9]:
import os
from dotenv import load_dotenv
load_dotenv()
from langchain_classic.chat_models import init_chat_model
os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")
llm = init_chat_model("groq:llama-3.1-8b-instant")


In [10]:
from langchain_core.output_parsers import StrOutputParser

In [14]:
#Function to get hypothetical documents
def get_hypo_documents(query):
    template = PromptTemplate.from_template(
    """Imagine you are an expert writing a detailed explanation on the topic: '{query}'
    create a hypothetical answer for the topic"""
    )
    hypothetical_answer_chain = template | llm | StrOutputParser()
    hypothetical_answer = hypothetical_answer_chain.invoke({"query": query})
    return hypothetical_answer

query = 'When was Steve Jobs fired from Apple?'
print(get_hypo_documents(query=query))


**The Firing of Steve Jobs: A Turning Point in Apple's History**

On September 29, 1985, Steve Jobs, the co-founder and then-CEO of Apple Inc., was abruptly fired from the company he helped create. This shocking event marked a significant turning point in Apple's history, leading to a period of turmoil and ultimately paving the way for the company's resurgence under Jobs' leadership.

**The Events Leading Up to the Firing**

In the early 1980s, Apple was experiencing tremendous growth and success, thanks in part to the introduction of the Macintosh computer in 1984. However, the company was also facing increasing competition from IBM and Microsoft, which was gaining traction with its Windows operating system.

As tensions rose within the company, Jobs' leadership style and vision for Apple began to clash with those of John Sculley, whom Jobs had hired as President in 1983. Sculley, a seasoned executive from PepsiCo, was more focused on maintaining a stable and profitable business model

In [16]:
matched_document = retriever.invoke(get_hypo_documents(query=query))
print(matched_document)

[Document(id='15c432c8-931b-4552-8cf5-dbea0edf6e1b', metadata={'title': 'Vicarious (company)', 'summary': 'Vicarious was an artificial intelligence company based in the San Francisco Bay Area, California. They use the theorized computational principles of the brain to attempt to build software that can think and learn like a human. Vicarious describes its technology as "a turnkey robotics solution integrator using artificial intelligence to automate tasks too complex and versatile for traditional automations". Alphabet Inc acquired the company in 2022 for an undisclosed amount.', 'source': 'https://en.wikipedia.org/wiki/Vicarious_(company)'}, page_content='Vicarious was an artificial intelligence company based in the San Francisco Bay Area, California. They use the theorized computational principles of the brain to attempt to build software that can think and learn like a human. Vicarious describes its technology as "a turnkey robotics solution integrator using artificial intelligence 

Langchain-HypotheticalDocumentEmbedder

In [17]:
from langchain_classic.chains.hyde.base import HypotheticalDocumentEmbedder

from langchain_classic.prompts import PromptTemplate
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_classic.document_loaders import TextLoader
from langchain_classic.text_splitter import RecursiveCharacterTextSplitter
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

In [18]:
# Step 1: Load and split documents
loader = TextLoader("langchain_crewai_dataset.txt")
docs = loader.load()
splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50)
chunks = splitter.split_documents(docs)

In [19]:
# Step 2: Set up LLM and embeddings

base_embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

C:\Users\Sakshi Sinha\AppData\Local\Temp\ipykernel_18444\140878275.py:3: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  base_embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1975.07it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [20]:
# Step 3: HyDE Embedder using prompt_key='web_search'
hyde_embedding_function = HypotheticalDocumentEmbedder.from_llm(
    llm=llm,
    base_embeddings=base_embeddings,
    prompt_key="web_search"
)

In [21]:
#step -4 Store documents in Chroma with HyDE embeddings
vector_store = Chroma.from_documents(
    documents= chunks,
    embedding= hyde_embedding_function,
    persist_directory="output/langchain"
)

In [22]:
# Step 5: RAG answer generation prompt
rag_prompt = PromptTemplate.from_template("""
Use the context below to answer the question.

Context:
{context}

Question: {input}
""")
rag_chain = create_stuff_documents_chain(llm=llm, prompt=rag_prompt)

In [23]:
#Step 6- RAG Final Pipeline
def hyde_rag_pipeline(query):
    matched_docs = vector_store.similarity_search(query, k=4)
    print(matched_docs)
    response = rag_chain.invoke(
        {
            "input" : query,
            "context" : matched_docs
        }
    )
    return response

In [24]:
# Step 7: Run example query
query = "What memory modules does LangChain provide?"
answer = hyde_rag_pipeline(query)
print("✅ Final Answer:\n", answer)

[Document(metadata={'source': 'langchain_crewai_dataset.txt'}, page_content='LangChain offers memory modules like ConversationBufferMemory and ConversationSummaryMemory. These allow the LLM to maintain awareness of previous conversation turns or summarize long interactions to fit within token limits. (v3)'), Document(metadata={'source': 'langchain_crewai_dataset.txt'}, page_content='LangChain offers memory modules like ConversationBufferMemory and ConversationSummaryMemory. These allow the LLM to maintain awareness of previous conversation turns or summarize long interactions to fit within token limits. (v10)'), Document(metadata={'source': 'langchain_crewai_dataset.txt'}, page_content='LangChain offers memory modules like ConversationBufferMemory and ConversationSummaryMemory. These allow the LLM to maintain awareness of previous conversation turns or summarize long interactions to fit within token limits. (v9)'), Document(metadata={'source': 'langchain_crewai_dataset.txt'}, page_cont